In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import sys
import cv2
import numpy as np
import multiprocessing as mp
import time
from pathlib    import Path
from tqdm       import tqdm  # для отображения прогресса, установите: pip install tqdm
from groundingdino.util.inference import load_model, load_image, predict

sys.path.append(os.path.abspath('..'))
from noises.wrapper import add_noise 


### Загрузка модели

In [2]:
config_path = "/Users/v.makshanchikov/Documents/Python Proj/ВКР Магистратура/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py"
weights_path = "/Users/v.makshanchikov/Documents/Python Proj/ВКР Магистратура/GroundingDINO/weights/groundingdino_swint_ogc.pth"

model = load_model(config_path, weights_path)
model = model.to('cpu')   # или 'cuda' если есть GPU

text_prompt = "people"
box_threshold = 0.35
text_threshold = 0.25

final text_encoder_type: bert-base-uncased


### Функции для последовательной обработки

In [3]:
def count_people_in_image(image_path):
    try:
        image_source, image = load_image(image_path)
        boxes, logits, phrases = predict(
            model=model,
            image=image,
            caption=text_prompt,
            box_threshold=box_threshold,
            text_threshold=text_threshold,
            device="cpu"
        )
        return len(boxes)
    except Exception as e:
        print(f"Ошибка {image_path}: {e}")
        return 0

def process_config(frame_dir, output_path, noise_params=None, skip_existing=True):
    """
    noise_params: dict для add_noise или None
    output_path: полный путь к файлу .npy
    """
    if skip_existing and output_path.exists():
        print(f"Пропуск (уже есть): {output_path}")
        return np.load(output_path)

    frame_files = sorted([f for f in os.listdir(frame_dir) if f.endswith('.jpg')])
    counts = []
    temp_dir = None

    if noise_params is not None:
        temp_dir = Path("temp_noisy_frames")
        temp_dir.mkdir(exist_ok=True)

    for frame_file in tqdm(frame_files, desc=f"→ {output_path.parent.name}/{output_path.name}"):
        img_path = os.path.join(frame_dir, frame_file)
        if noise_params is not None:
            img = cv2.imread(img_path)
            if img is None:
                counts.append(0)
                continue
            noisy_img = add_noise(img, **noise_params)
            temp_path = temp_dir / frame_file
            cv2.imwrite(str(temp_path), noisy_img)
            cnt = count_people_in_image(str(temp_path))
        else:
            cnt = count_people_in_image(img_path)
        counts.append(cnt)

    if temp_dir is not None:
        for f in temp_dir.iterdir():
            f.unlink()
        temp_dir.rmdir()

    counts = np.array(counts, dtype=int)
    np.save(output_path, counts)
    print(f"Сохранено: {output_path}")
    return counts

def run_experiments():
    OUTPUT_BASE.mkdir(exist_ok=True)

    # Baseline
    orig_path = OUTPUT_BASE / "original.npy"
    process_config(FRAME_DIR, orig_path, noise_params=None, skip_existing=True)

    # ---- Конфигурации экспериментов ----
    experiments_config = [
        {
            "noise_type": "gaussian",
            "param_name": "var",
            "values": np.linspace(0.001, 0.05, 10),
            "fixed_params": {"mean": 0.0}
        },
        {
            "noise_type": "salt_pepper",
            "param_name": "prob",
            "values": np.linspace(0.001, 0.05, 10),
            "fixed_params": {}
        },
        {
            "noise_type": "speckle",
            "param_name": "scale",
            "values": np.linspace(0.05, 0.8, 12),
            "fixed_params": {}
        },
        {
            "noise_type": "poisson",
            "param_name": "scale",
            "values": np.linspace(0.2, 2.0, 10),
            "fixed_params": {}
        }
    ]

    for cfg in experiments_config:
        noise_type = cfg["noise_type"]
        param_name = cfg["param_name"]
        values = cfg["values"]
        fixed = cfg["fixed_params"]

        for val in values:
            # Формируем параметры для add_noise
            if noise_type == "gaussian":
                params = {
                    "noise_type": "gaussian",
                    "mean": fixed.get("mean", 0.0),
                    "var": val
                }
                filename = f"{noise_type}_{param_name}_{val:.6f}.npy"
            elif noise_type == "salt_pepper":
                params = {
                    "noise_type": "salt_pepper",
                    "salt_prob": val,
                    "pepper_prob": val
                }
                filename = f"{noise_type}_{param_name}_{val:.6f}.npy"
            elif noise_type == "speckle":
                params = {
                    "noise_type": "speckle",
                    "speckle_scale": val
                }
                filename = f"{noise_type}_{param_name}_{val:.6f}.npy"
            elif noise_type == "poisson":
                params = {
                    "noise_type": "poisson",
                    "poisson_scale": val
                }
                filename = f"{noise_type}_{param_name}_{val:.6f}.npy"
            else:
                continue

            out_path = OUTPUT_BASE / filename
            process_config(FRAME_DIR, out_path, noise_params=params, skip_existing=True)

    print("\nВсе эксперименты завершены!")

### Последовательный эксперимент

In [4]:
FRAME_DIR = "/Users/v.makshanchikov/Documents/Python Proj/ВКР Магистратура/mall_dataset/pre_frames"
OUTPUT_BASE = Path("/Users/v.makshanchikov/Documents/Python Proj/ВКР Магистратура/inferense_datasets/groundingdino_results_cons")
OUTPUT_BASE.mkdir(exist_ok=True)

start = time.perf_counter()
run_experiments()
end = time.perf_counter()
print(f"Общее время выполнения: {end - start:.2f} сек")

→ groundingdino_results_cons/original.npy: 100%|██████████| 2/2 [00:07<00:00,  3.67s/it]


Сохранено: /Users/v.makshanchikov/Documents/Python Proj/ВКР Магистратура/inferense_datasets/groundingdino_results_cons/original.npy


→ groundingdino_results_cons/gaussian_var_0.001000.npy: 100%|██████████| 2/2 [00:07<00:00,  3.55s/it]


Сохранено: /Users/v.makshanchikov/Documents/Python Proj/ВКР Магистратура/inferense_datasets/groundingdino_results_cons/gaussian_var_0.001000.npy


→ groundingdino_results_cons/gaussian_var_0.006444.npy: 100%|██████████| 2/2 [00:07<00:00,  3.57s/it]


...

Сохранено: /Users/v.makshanchikov/Documents/Python Proj/ВКР Магистратура/inferense_datasets/groundingdino_results_cons/poisson_scale_1.600000.npy


→ groundingdino_results_cons/poisson_scale_1.800000.npy: 100%|██████████| 2/2 [00:08<00:00,  4.19s/it]


Сохранено: /Users/v.makshanchikov/Documents/Python Proj/ВКР Магистратура/inferense_datasets/groundingdino_results_cons/poisson_scale_1.800000.npy


→ groundingdino_results_cons/poisson_scale_2.000000.npy: 100%|██████████| 2/2 [00:08<00:00,  4.24s/it]

Сохранено: /Users/v.makshanchikov/Documents/Python Proj/ВКР Магистратура/inferense_datasets/groundingdino_results_cons/poisson_scale_2.000000.npy

Все эксперименты завершены!
Общее время выполнения: 336.51 сек


### Считывание файлов

In [ ]:
import re

results_dir = Path("groundingdino_results")

# 1. Загружаем оригинал
original = np.load(results_dir / "original.npy")

# 2. Собираем все файлы гауссовского шума
gaussian_files = list(results_dir.glob("gaussian_var_*.npy"))

# 3. Извлекаем значения параметра var из имени
data = []
for f in gaussian_files:
    # Извлекаем число после "var_"
    match = re.search(r"var_([0-9.]+)\.npy", f.name)
    if match:
        var = float(match.group(1))
        counts = np.load(f)
        data.append((var, counts))

# 4. Сортируем по var
data.sort(key=lambda x: x[0])

# 5. Для каждого var вычисляем метрику (например, среднюю абсолютную ошибку counts)
for var, counts in data:
    mae = np.mean(np.abs(counts - original))
    print(f"var={var:.6f}, MAE={mae:.3f}")